In [ ]:
from google.colab import drive
import os

# 1. Connect to Google Drive
drive.mount('/content/drive')

# 2. Extract the dataset from Drive into Colab's  local storage
print("Extracting dataset...")
# Make sure the path matches
!tar -xzf "/content/drive/MyDrive/XAI_Project/CUB_200_2011.tgz" -C /content/

# 3. Verify it worked
dataset_path = '/content/CUB_200_2011'
if os.path.exists(dataset_path):
    print("Success! Dataset extracted.")
    print("Folders inside:", os.listdir(dataset_path))
else:
    print("Something went wrong. Check your file paths.")

Mounted at /content/drive
Extracting dataset...
Success! Dataset extracted.
Folders inside: ['attributes', 'README', 'parts', 'images', 'bounding_boxes.txt', 'images.txt', 'classes.txt', 'train_test_split.txt', 'image_class_labels.txt']


In [ ]:
import pandas as pd
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import torchvision.models as models
import torch.nn as nn

# Define the custom Dataset
class CUB200Dataset(Dataset):
    def __init__(self, root_dir, is_train=True, transform=None):
        self.root_dir = root_dir
        self.transform = transform

        images = pd.read_csv(os.path.join(root_dir, 'images.txt'), sep=' ', names=['img_id', 'filepath'])
        labels = pd.read_csv(os.path.join(root_dir, 'image_class_labels.txt'), sep=' ', names=['img_id', 'target'])
        splits = pd.read_csv(os.path.join(root_dir, 'train_test_split.txt'), sep=' ', names=['img_id', 'is_training_image'])

        self.data = images.merge(labels, on='img_id').merge(splits, on='img_id')
        split_val = 1 if is_train else 0
        self.data = self.data[self.data['is_training_image'] == split_val].reset_index(drop=True)
        self.data['target'] = self.data['target'] - 1

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        img_path = os.path.join(self.root_dir, 'images', row['filepath'])
        image = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        return image, row['target']

# Set up transforms and DataLoaders
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

test_dataset = CUB200Dataset(root_dir='/content/CUB_200_2011', is_train=False, transform=transform)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=2)

print(f"Test dataset loaded with {len(test_dataset)} images.")

# Load Model onto GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, 200)
model = model.to(device)

print("ResNet-50 initialized and moved to GPU!")

Test dataset loaded with 5794 images.
Using device: cuda
Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 113MB/s]


ResNet-50 initialized and moved to GPU!


In [ ]:
import torch
from tqdm import tqdm

def validate_baseline(model, dataloader, device):
    print("Running validation check...")
    correct = 0
    total = 0

    # Put model in evaluation mode (turn off training-specific layers like Dropout)
    model.eval()


    with torch.no_grad():
        # progress bar
        for images, labels in tqdm(dataloader, desc="Testing Images"):
            images, labels = images.to(device), labels.to(device)

            # 1. Pass images through the model
            outputs = model(images)

            # 2. Get the highest probability prediction
            _, predicted = torch.max(outputs.data, 1)

            # 3. Scores
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    accuracy = 100 * correct / total
    print(f'\nBaseline Validation Accuracy: {accuracy:.2f}%')
    print("Day 1 Complete! The model needs fine-tuning before XAI can be applied.")

# Run the function
validate_baseline(model, test_loader, device)

Running validation check...


Testing Images: 100%|██████████| 182/182 [00:35<00:00,  5.15it/s]


Baseline Validation Accuracy: 0.64%
Day 1 Complete! The model needs fine-tuning before XAI can be applied.


In [ ]:
import torch.optim as optim
from tqdm import tqdm

# 1. Load the TRAINING data this time
train_dataset = CUB200Dataset(root_dir='/content/CUB_200_2011', is_train=True, transform=transform)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)
print(f"Training dataset loaded with {len(train_dataset)} images.")

# 2. Set up the Optimizer and Loss Function
criterion = nn.CrossEntropyLoss()
# We use a small learning rate

# 3. Training Loop
epochs = 10
print("Starting Fine-Tuning...")

for epoch in range(epochs):
    model.train() # Set model to training mode
    running_loss = 0.0
    correct = 0
    total = 0

    # Progress bar for the training loop
    progress = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}")
    for images, labels in progress:
        images, labels = images.to(device), labels.to(device)

        # Zero the gradients
        optimizer.zero_grad()

        # Forward pass
        outputs = model(images)
        loss = criterion(outputs, labels)

        # Backward pass and optimize
        loss.backward()
        optimizer.step()

        # Calculate training accuracy
        running_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

        progress.set_postfix(loss=running_loss/total, accuracy=100.*correct/total)

print("\nFinished Fine-Tuning. Saving CUB-200 model weights...")

# 4. Save the high-accuracy weights to your Google Drive!
save_path = '/content/drive/MyDrive/XAI_Project/resnet50_cub200_finetuned.pth'
torch.save(model.state_dict(), save_path)
print(f"Model saved to: {save_path}")

Training dataset loaded with 5994 images.
Starting Fine-Tuning...


Epoch 10/10: 100%|██████████| 188/188 [01:16<00:00,  2.45it/s, accuracy=99.8, loss=0.000824]



Finished Fine-Tuning! Saving your CUB-200 model weights...
Model saved to: /content/drive/MyDrive/XAI_Project/resnet50_cub200_finetuned.pth


In [ ]:
import torch
from tqdm import tqdm

def validate_baseline(model, dataloader, device):
    print("Running validation check...")
    correct = 0
    total = 0

    # Put the model in evaluation mode (turns off training-specific layers like Dropout)
    model.eval()

    # We don't need to calculate gradients for just testing
    with torch.no_grad():
        # progress bar!
        for images, labels in tqdm(dataloader, desc="Testing Images"):
            images, labels = images.to(device), labels.to(device)

            # 1. Pass images through the model
            outputs = model(images)

            # 2. Get the highest probability prediction
            _, predicted = torch.max(outputs.data, 1)

            # 3. Tally up the score
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    accuracy = 100 * correct / total
    print(f'\nBaseline Validation Accuracy: {accuracy:.2f}%')
    print("Day 1 Complete! The model needs fine-tuning before XAI can be applied.")

# Run the function
validate_baseline(model, test_loader, device)

Running validation check...


Testing Images: 100%|██████████| 182/182 [00:38<00:00,  4.76it/s]


Baseline Validation Accuracy: 71.00%
Day 1 Complete! The model needs fine-tuning before XAI can be applied.


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
from tqdm import tqdm

# 1. Define Training Transforms WITH Data Augmentation
# We resize it slightly larger first, then randomly crop it to 224x224.
# This forces the model to look at different parts of the bird every time
train_transform_aug = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomCrop((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5), # 50% chance to mirror the image
    transforms.RandomRotation(degrees=15),  # Tilt the image slightly
    transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1), # Slight lighting changes
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# 2. Reload the Training Dataset with the new augmented transforms
train_dataset = CUB200Dataset(root_dir='/content/CUB_200_2011', is_train=True, transform=train_transform_aug)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)
print(f"Augmented Training dataset loaded with {len(train_dataset)} images.")

# 3. Setup Optimizer and Learning Rate Scheduler
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.0001)

# The Scheduler: This will cut the learning rate in half every 7 epochs.
# This allows the model to make large adjustments early on, and tiny, precise adjustments later.
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=7, gamma=0.5)

# 4. The Updated Training Loop
epochs = 20  # Bumping up to 20 epochs since augmentation prevents quick overfitting
print("Starting Upgraded Fine-Tuning...")

for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    # Display the current Learning Rate in the progress bar so you can watch it drop
    current_lr = scheduler.get_last_lr()[0]
    progress = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} [LR: {current_lr:.6f}]")

    for images, labels in progress:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

        progress.set_postfix(loss=running_loss/total, accuracy=100.*correct/total)

    # Step the scheduler AFTER each epoch finishes
    scheduler.step()

print("\nFinished Upgraded Fine-Tuning!")

# 5. Overwrite previous saved weights with this new, smarter version
save_path = '/content/drive/MyDrive/XAI_Project/resnet50_cub200_finetuned.pth'
torch.save(model.state_dict(), save_path)
print(f"Upgraded model saved to: {save_path}")

Augmented Training dataset loaded with 5994 images.
Starting Upgraded Fine-Tuning...


Epoch 20/20 [LR: 0.000025]: 100%|██████████| 188/188 [01:17<00:00,  2.42it/s, accuracy=99.9, loss=0.000346]



Finished Upgraded Fine-Tuning!
Upgraded model saved to: /content/drive/MyDrive/XAI_Project/resnet50_cub200_finetuned.pth


In [ ]:
import torch
from tqdm import tqdm



# Run the function
validate_baseline(model, test_loader, device)

Running validation check...


Testing Images: 100%|██████████| 182/182 [00:36<00:00,  4.97it/s]


Baseline Validation Accuracy: 76.20%
Day 1 Complete! The model needs fine-tuning before XAI can be applied.


In [ ]:
import torch.nn.functional as F
import pandas as pd
from tqdm import tqdm


def save_highly_confident_subset(model, dataloader, device, top_k_per_class=3):
    print("Evaluating model confidence across the test set...")
    model.eval() # Ensure model is in eval mode

    results = []

    with torch.no_grad():
        # We need to track the actual dataset index
        for batch_idx, (images, labels) in enumerate(tqdm(dataloader, desc="Scoring Confidence")):
            images, labels = images.to(device), labels.to(device)

            # Get raw outputs and convert to probabilities (0.0 to 1.0)
            logits = model(images)
            probs = F.softmax(logits, dim=1)

            # For each image in the batch, record how confident it was about the CORRECT answer
            for i in range(len(labels)):
                true_label = labels[i].item()
                prob_of_true_class = probs[i, true_label].item()

                # Calculate the image's original index in the full test dataset
                global_idx = batch_idx * dataloader.batch_size + i

                results.append({
                    'dataset_idx': global_idx,
                    'true_label': true_label,
                    'confidence': prob_of_true_class
                })

    # Convert our results to a Pandas DataFrame for easy sorting
    df = pd.DataFrame(results)

    # 1. Sort all predictions by confidence (highest first)
    # 2. Group them by the bird class
    # 3. Skim the top 'k' from each group
    top_confident_df = df.sort_values('confidence', ascending=False) \
                         .groupby('true_label') \
                         .head(top_k_per_class)

    return top_confident_df

# Run the function: 3 images * 200 classes = 600 total images
confident_subset_df = save_highly_confident_subset(model, test_loader, device, top_k_per_class=3)

# Save this subset to Drive
csv_path = '/content/drive/MyDrive/XAI_Project/confident_subset.csv'
confident_subset_df.to_csv(csv_path, index=False)

print(f"\nSuccess! Saved a curated list of {len(confident_subset_df)} highly confident images to: {csv_path}")

Evaluating model confidence across the test set...


Scoring Confidence: 100%|██████████| 182/182 [00:34<00:00,  5.27it/s]


Success! Saved a curated list of 600 highly confident images to: /content/drive/MyDrive/XAI_Project/confident_subset.csv
